In [1]:
import numpy as np
import os

# ── Edit these two lines to point at your session ─────────────────────────
SESSION_DIR  = r'20260625_exp1_g0'
SESSION_NAME = '20260625_exp1_g0'
# ──────────────────────────────────────────────────────────────────────────

NIDQ_BIN       = os.path.join(SESSION_DIR, f'{SESSION_NAME}_t0.nidq.bin')
NIDQ_META_PATH = os.path.join(SESSION_DIR, f'{SESSION_NAME}_t0.nidq.meta')
IMEC_BIN       = os.path.join(SESSION_DIR, f'{SESSION_NAME}_imec0',
                               f'{SESSION_NAME}_t0.imec0.ap.bin')
IMEC_META_PATH = os.path.join(SESSION_DIR, f'{SESSION_NAME}_imec0',
                               f'{SESSION_NAME}_t0.imec0.ap.meta')

def read_meta(path):
    meta = {}
    with open(path, 'r') as f:
        for line in f:
            if '=' in line:
                k, v = line.strip().split('=', 1)
                meta[k] = v
    return meta

# ── NIDQ ──────────────────────────────────────────────────────────────────
nidq_meta = read_meta(NIDQ_META_PATH)
fs_ni     = float(nidq_meta['niSampRate'])
nc_ni     = int(nidq_meta['nSavedChans'])        # 2: XA0 (analog LED) + XD0 (digital sync)
first_ni  = int(nidq_meta['firstSample'])        # absolute sample offset

nidq_mm   = np.memmap(NIDQ_BIN, dtype=np.int16, mode='r').reshape(-1, nc_ni)
n_ni      = len(nidq_mm)

# XA0 (ch 0) = LED analog signal; XD0 (ch 1) = sync pulse (not used here)
v_per_bit = float(nidq_meta['niAiRangeMax']) / float(nidq_meta['niMaxInt'])
led       = nidq_mm[:, 0].astype(np.float32) * v_per_bit   # volts

# ── IMEC meta (for timebase alignment) ────────────────────────────────────
imec_meta  = read_meta(IMEC_META_PATH)
fs_im      = float(imec_meta['imSampRate'])
first_im   = int(imec_meta['firstSample'])
nc_im      = int(imec_meta['nSavedChans'])       # 385: 384 AP + 1 SY
n_im       = os.path.getsize(IMEC_BIN) // 2 // nc_im

# ── Align NIDQ time to IMEC-relative time ─────────────────────────────────
# Both streams share the same absolute clock (syncSourceIdx=3).
# Convert NIDQ absolute sample numbers to IMEC-relative seconds.
imec_t0_abs = first_im / fs_im
t_led_imec  = (np.arange(n_ni) + first_ni) / fs_ni - imec_t0_abs

# ── LED onset detection ────────────────────────────────────────────────────
led_thresh      = (led.max() + led.min()) / 2
led_onsets_imec = t_led_imec[np.where(np.diff((led > led_thresh).astype(np.int8)) > 0)[0] + 1]

print(f'NIDQ : {n_ni} samples @ {fs_ni:.1f} Hz  ({n_ni/fs_ni:.2f} s)')
print(f'IMEC : {n_im} samples @ {fs_im:.0f} Hz  ({n_im/fs_im:.3f} s)')
print(f'LED range : {led.min():.3f} V – {led.max():.3f} V  |  threshold = {led_thresh:.3f} V')
print(f'LED onsets detected : {len(led_onsets_imec)}')
if len(led_onsets_imec):
    print(f'First 5 onsets (s)  : {led_onsets_imec[:5]}')


NIDQ : 57153973 samples @ 11574.1 Hz  (4938.09 s)
IMEC : 148142489 samples @ 30000 Hz  (4938.083 s)
LED range : -0.073 V – 4.010 V  |  threshold = 1.968 V
LED onsets detected : 9523
First 5 onsets (s)  : [35.74268758 35.74311958 35.74329238 35.74372438 35.74398358]


In [2]:
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, TextBox

# BNC/LED-only viewer — use this instead of the cell below when the IMEC
# recording is too large to memmap. It only needs t_led_imec / led /
# led_onsets_imec from the NIDQ cell above (no IMEC .bin access at all).

_MAX_DISPLAY_PTS = 10_000


def _downsample(t, y, max_pts=_MAX_DISPLAY_PTS):
    n = len(t)
    if n <= max_pts:
        return t, y
    s = n // max_pts
    return t[::s], y[::s]


class BNCTraceViewer:
    def __init__(self, t_led_imec, led, led_onsets_imec, window_sec=5.0):
        self.t_led  = t_led_imec
        self.led    = led
        self.onsets = led_onsets_imec
        self.dur    = float(t_led_imec[-1]) if len(t_led_imec) else 0.0

        self.window_sec   = min(window_sec, max(self.dur, 0.05))
        self.t0           = 0.0
        self._tb_updating = False
        self._key_cid     = None

        self._build()

    def _build(self):
        self.fig, self.ax_led = plt.subplots(figsize=(16, 4))
        self.fig.subplots_adjust(bottom=0.28, top=0.93)

        ax_st = self.fig.add_axes([0.10, 0.14, 0.60, 0.04])
        ax_sw = self.fig.add_axes([0.10, 0.07, 0.60, 0.04])

        self.sl_time = Slider(ax_st, 'Time (s)', 0,
                              max(self.dur - self.window_sec, 0.05),
                              valinit=self.t0, valstep=0.05)
        self.sl_win  = Slider(ax_sw, 'Window (s)', 0.05,
                              max(self.dur, 0.1),
                              valinit=self.window_sec, valstep=0.05)

        ax_tb_t = self.fig.add_axes([0.73, 0.135, 0.10, 0.045])
        ax_tb_w = self.fig.add_axes([0.73, 0.065, 0.10, 0.045])

        self.tb_time = TextBox(ax_tb_t, '', initial=f'{self.t0:.2f}')
        self.tb_win  = TextBox(ax_tb_w, '', initial=f'{self.window_sec:.2f}')
        self._textboxes = (self.tb_time, self.tb_win)

        for tb in self._textboxes:
            _orig_begin = tb.begin_typing
            _orig_stop  = tb.stop_typing

            def _begin(*args, orig=_orig_begin):
                orig(*args)
                if self._key_cid is not None:
                    self.fig.canvas.mpl_disconnect(self._key_cid)
                    self._key_cid = None

            def _stop(*args, orig=_orig_stop):
                orig(*args)
                if self._key_cid is None:
                    self._key_cid = self.fig.canvas.mpl_connect(
                        'key_press_event', self._on_key)

            tb.begin_typing = _begin
            tb.stop_typing  = _stop

        self.sl_time.on_changed(self._on_time)
        self.sl_win.on_changed(self._on_window)
        self.tb_time.on_submit(self._on_tb_time)
        self.tb_win.on_submit(self._on_tb_win)

        self._key_cid = self.fig.canvas.mpl_connect('key_press_event', self._on_key)
        self.fig.text(0.01, 0.02, '← → scroll time   + − zoom window',
                      fontsize=7, color='gray', va='bottom')

        self._draw()
        plt.show()

    def _draw(self):
        t0 = self.t0
        t1 = t0 + self.window_sec
        onsets_in_view = self.onsets[(self.onsets >= t0) & (self.onsets <= t1)]

        mask = (self.t_led >= t0) & (self.t_led <= t1)
        t_led_v, led_v = _downsample(self.t_led[mask], self.led[mask])

        self.ax_led.cla()
        self.ax_led.plot(t_led_v, led_v, color='cyan', lw=0.8)
        for o in onsets_in_view:
            self.ax_led.axvline(o, color='yellow', alpha=0.6, lw=1)
        led_max = float(self.led.max()) if self.led.max() > 0 else 1.0
        self.ax_led.set_ylim(-0.1, led_max * 1.3)
        self.ax_led.set_ylabel('LED / BNC (V)')
        self.ax_led.set_xlabel('Time (s)')
        self.ax_led.set_facecolor('#1a1a2e')
        self.ax_led.set_xlim(t0, t1)
        self.ax_led.set_title(f'{len(onsets_in_view)} onset(s) in view  |  '
                              f'{len(self.onsets)} total', fontsize=9)
        self.fig.canvas.draw_idle()

    def _on_time(self, val):
        self.t0 = val
        self._sync_tb(self.tb_time, f'{val:.2f}')
        self._draw()

    def _on_window(self, val):
        self.window_sec = val
        new_max = max(self.dur - val, 0.05)
        self.sl_time.valmax = new_max
        self.sl_time.ax.set_xlim(0, new_max)
        self.t0 = min(self.t0, new_max)
        self._sync_tb(self.tb_win, f'{val:.2f}')
        self._draw()

    def _sync_tb(self, tb, text):
        self._tb_updating = True
        tb.set_val(text)
        self._tb_updating = False

    def _on_tb_time(self, text):
        if self._tb_updating:
            return
        try:
            self.sl_time.set_val(np.clip(float(text), 0, self.sl_time.valmax))
        except ValueError:
            self._sync_tb(self.tb_time, f'{self.t0:.2f}')

    def _on_tb_win(self, text):
        if self._tb_updating:
            return
        try:
            self.sl_win.set_val(np.clip(float(text), 0.05, self.sl_win.valmax))
        except ValueError:
            self._sync_tb(self.tb_win, f'{self.window_sec:.2f}')

    def _on_key(self, event):
        if event.key == 'right':
            new_t = min(self.t0 + self.window_sec * 0.5, self.sl_time.valmax)
            self.sl_time.set_val(np.clip(new_t, 0, self.sl_time.valmax))
        elif event.key == 'left':
            self.sl_time.set_val(np.clip(max(self.t0 - self.window_sec * 0.5, 0.0),
                                         0, self.sl_time.valmax))
        elif event.key in ('+', '='):
            self.sl_win.set_val(max(0.05, self.window_sec * 0.5))
        elif event.key == '-':
            self.sl_win.set_val(min(self.sl_win.valmax, self.window_sec * 2.0))


# ── Launch (BNC-only — no IMEC memmap needed) ───────────────────────────────
bnc_viewer = BNCTraceViewer(
    t_led_imec      = t_led_imec,
    led             = led,
    led_onsets_imec = led_onsets_imec,
    window_sec      = 5.0,
)


Traceback (most recent call last):
  File "c:\miniconda\envs\spikesort\Lib\site-packages\matplotlib\cbook.py", line 390, in process
    func(*args, **kwargs)
  File "c:\miniconda\envs\spikesort\Lib\site-packages\matplotlib\widgets.py", line 184, in wrapper
    if event.inaxes is not self.ax:
       ^^^^^^^^^^^^
AttributeError: 'ResizeEvent' object has no attribute 'inaxes'
Traceback (most recent call last):
  File "c:\miniconda\envs\spikesort\Lib\site-packages\matplotlib\cbook.py", line 390, in process
    func(*args, **kwargs)
  File "c:\miniconda\envs\spikesort\Lib\site-packages\matplotlib\widgets.py", line 184, in wrapper
    if event.inaxes is not self.ax:
       ^^^^^^^^^^^^
AttributeError: 'ResizeEvent' object has no attribute 'inaxes'


In [2]:
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, TextBox
import matplotlib.gridspec as gridspec

_MAX_DISPLAY_PTS = 10_000   # max samples sent to renderer per trace


def _downsample(t, y, max_pts=_MAX_DISPLAY_PTS):
    """Stride-decimate (t, y) to at most max_pts for display."""
    n = len(t)
    if n <= max_pts:
        return t, y
    s = n // max_pts
    return t[::s], y[::s]


class NeuralLEDViewer:
    def __init__(self,
                 imec_bin, imec_meta_path,
                 t_led_imec, led, led_onsets_imec,
                 n_channels=4,
                 ch_start=0,
                 window_sec=1.0):

        meta       = read_meta(imec_meta_path)
        self.fs    = float(meta['imSampRate'])
        self.nc    = int(meta['nSavedChans'])
        self.mm    = np.memmap(imec_bin, dtype=np.int16, mode='r').reshape(-1, self.nc)
        self.dur   = len(self.mm) / self.fs
        self.uv_per_bit = float(meta['imAiRangeMax']) * 1e6 / float(meta['imMaxInt']) / 80.0
        self.n_ap  = self.nc - 1

        self.t_led   = t_led_imec
        self.led     = led
        self.onsets  = led_onsets_imec

        self.n_channels  = n_channels
        self.ch_start    = ch_start
        self.window_sec  = window_sec
        self.t0          = 0.0
        self._tb_updating = False
        self._key_cid     = None

        print(f'uV/bit={self.uv_per_bit:.4f}  dur={self.dur:.2f}s  AP channels=0–{self.n_ap-1}')
        self._build()

    # ── Build figure ───────────────────────────────────────────────────────
    def _build(self):
        n = self.n_channels
        row_heights = [1.0] + [3.0] * n
        self.fig = plt.figure(figsize=(16, 2 + sum(row_heights) * 0.8))
        gs = gridspec.GridSpec(len(row_heights), 1,
                               height_ratios=row_heights,
                               hspace=0.05, top=0.95, bottom=0.20)

        self.ax_led = self.fig.add_subplot(gs[0])
        self.ax_raw = []
        for i in range(n):
            self.ax_raw.append(self.fig.add_subplot(gs[1 + i], sharex=self.ax_led))

        ax_st = self.fig.add_axes([0.10, 0.13, 0.60, 0.02])
        ax_sc = self.fig.add_axes([0.10, 0.08, 0.60, 0.02])
        ax_sw = self.fig.add_axes([0.10, 0.03, 0.60, 0.02])

        self.sl_time = Slider(ax_st, 'Time (s)',   0,
                              max(self.dur - self.window_sec, 0.05),
                              valinit=self.t0, valstep=0.05)
        self.sl_ch   = Slider(ax_sc, 'Channel',    0,
                              max(self.n_ap - self.n_channels, 0),
                              valinit=self.ch_start, valstep=1)
        self.sl_win  = Slider(ax_sw, 'Window (s)', 0.05, 30.0,
                              valinit=self.window_sec, valstep=0.05)

        ax_tb_t = self.fig.add_axes([0.73, 0.115, 0.09, 0.040])
        ax_tb_c = self.fig.add_axes([0.73, 0.065, 0.09, 0.040])
        ax_tb_w = self.fig.add_axes([0.73, 0.015, 0.09, 0.040])

        self.tb_time = TextBox(ax_tb_t, '', initial=f'{self.t0:.2f}')
        self.tb_ch   = TextBox(ax_tb_c, '', initial=str(self.ch_start))
        self.tb_win  = TextBox(ax_tb_w, '', initial=f'{self.window_sec:.2f}')
        self._textboxes = (self.tb_time, self.tb_ch, self.tb_win)

        for tb in self._textboxes:
            _orig_begin = tb.begin_typing
            _orig_stop  = tb.stop_typing

            def _begin(*args, orig=_orig_begin):
                orig(*args)
                if self._key_cid is not None:
                    self.fig.canvas.mpl_disconnect(self._key_cid)
                    self._key_cid = None

            def _stop(*args, orig=_orig_stop):
                orig(*args)
                if self._key_cid is None:
                    self._key_cid = self.fig.canvas.mpl_connect(
                        'key_press_event', self._on_key)

            tb.begin_typing = _begin
            tb.stop_typing  = _stop

        self.sl_time.on_changed(self._on_time)
        self.sl_ch.on_changed(self._on_channel)
        self.sl_win.on_changed(self._on_window)
        self.tb_time.on_submit(self._on_tb_time)
        self.tb_ch.on_submit(self._on_tb_ch)
        self.tb_win.on_submit(self._on_tb_win)

        self._key_cid = self.fig.canvas.mpl_connect('key_press_event', self._on_key)
        self.fig.text(0.01, 0.155, '← → scroll time   ↑ ↓ scroll channels   + − zoom window',
                      fontsize=7, color='gray', va='bottom')

        self._draw()
        plt.show()

    # ── Draw ───────────────────────────────────────────────────────────────
    def _draw(self):
        t0 = self.t0
        t1 = t0 + self.window_sec
        s0 = int(t0 * self.fs)
        s1 = int(t1 * self.fs)
        onsets_in_view = self.onsets[(self.onsets >= t0) & (self.onsets <= t1)]

        # LED panel — downsample so sine shape is preserved
        mask = (self.t_led >= t0) & (self.t_led <= t1)
        t_led_v, led_v = _downsample(self.t_led[mask], self.led[mask])
        self.ax_led.cla()
        self.ax_led.plot(t_led_v, led_v, color='cyan', lw=0.8)
        for o in onsets_in_view:
            self.ax_led.axvline(o, color='yellow', alpha=0.6, lw=1)
        led_max = float(self.led.max()) if self.led.max() > 0 else 1.0
        self.ax_led.set_ylim(-0.1, led_max * 1.3)
        self.ax_led.set_ylabel('LED', fontsize=9)
        self.ax_led.set_facecolor('#1a1a2e')
        plt.setp(self.ax_led.get_xticklabels(), visible=False)

        # AP panels — downsample each trace the same way
        t_ap_raw = np.arange(s0, s1) / self.fs
        channels = list(range(self.ch_start,
                               min(self.ch_start + self.n_channels, self.n_ap)))

        for i, ax_r in enumerate(self.ax_raw):
            if i < len(channels):
                ch = channels[i]
                raw = self.mm[s0:s1, ch].astype(np.float32) * self.uv_per_bit
                t_ds, y_ds = _downsample(t_ap_raw, raw)
                ax_r.cla()
                ax_r.plot(t_ds, y_ds, color='white', lw=0.5)
                for o in onsets_in_view:
                    ax_r.axvline(o, color='cyan', alpha=0.35, lw=1, ls='--')
                ax_r.set_ylabel(f'AP{ch}\n(µV)', fontsize=8)
                ax_r.set_facecolor('#1a1a2e')
            else:
                ax_r.cla(); ax_r.set_visible(False)

            plt.setp(ax_r.get_xticklabels(), visible=(i == self.n_channels - 1))

        self.ax_raw[-1].set_xlabel('Time (s)')
        self.ax_led.set_xlim(t0, t1)
        self.fig.canvas.draw_idle()

    # ── Slider callbacks ───────────────────────────────────────────────────
    def _on_time(self, val):
        self.t0 = val
        self._sync_tb(self.tb_time, f'{val:.2f}')
        self._draw()

    def _on_channel(self, val):
        self.ch_start = int(round(val))
        self._sync_tb(self.tb_ch, str(self.ch_start))
        self._draw()

    def _on_window(self, val):
        self.window_sec = val
        new_max = max(self.dur - val, 0.05)
        self.sl_time.valmax = new_max
        self.sl_time.ax.set_xlim(0, new_max)
        self.t0 = min(self.t0, new_max)
        self._sync_tb(self.tb_win, f'{val:.2f}')
        self._draw()

    def _sync_tb(self, tb, text):
        self._tb_updating = True
        tb.set_val(text)
        self._tb_updating = False

    # ── TextBox callbacks ──────────────────────────────────────────────────
    def _on_tb_time(self, text):
        if self._tb_updating:
            return
        try:
            self.sl_time.set_val(np.clip(float(text), 0, self.sl_time.valmax))
        except ValueError:
            self._sync_tb(self.tb_time, f'{self.t0:.2f}')

    def _on_tb_ch(self, text):
        if self._tb_updating:
            return
        try:
            self.sl_ch.set_val(np.clip(int(round(float(text))), 0,
                                       self.n_ap - self.n_channels))
        except ValueError:
            self._sync_tb(self.tb_ch, str(self.ch_start))

    def _on_tb_win(self, text):
        if self._tb_updating:
            return
        try:
            self.sl_win.set_val(np.clip(float(text), 0.05, 30.0))
        except ValueError:
            self._sync_tb(self.tb_win, f'{self.window_sec:.2f}')

    # ── Keyboard controls ──────────────────────────────────────────────────
    def _on_key(self, event):
        if event.key == 'right':
            new_t = min(self.t0 + self.window_sec * 0.5, self.dur - self.window_sec)
            self.sl_time.set_val(np.clip(new_t, 0, self.sl_time.valmax))
        elif event.key == 'left':
            self.sl_time.set_val(np.clip(max(self.t0 - self.window_sec * 0.5, 0.0),
                                         0, self.sl_time.valmax))
        elif event.key == 'down':
            self.sl_ch.set_val(min(self.ch_start + 1, self.n_ap - self.n_channels))
        elif event.key == 'up':
            self.sl_ch.set_val(max(self.ch_start - 1, 0))
        elif event.key in ('+', '='):
            self.sl_win.set_val(max(0.05, self.window_sec * 0.5))
        elif event.key == '-':
            self.sl_win.set_val(min(30.0, self.window_sec * 2.0))


# ── Launch ─────────────────────────────────────────────────────────────────
viewer = NeuralLEDViewer(
    imec_bin        = IMEC_BIN,
    imec_meta_path  = IMEC_META_PATH,
    t_led_imec      = t_led_imec,
    led             = led,
    led_onsets_imec = led_onsets_imec,
    n_channels  = 4,
    ch_start    = 0,
    window_sec  = 1.0,
)


uV/bit=3.7842  dur=68.36s  AP channels=0–383


Traceback (most recent call last):
  File "c:\miniconda\envs\spikesort\Lib\site-packages\matplotlib\cbook.py", line 390, in process
    func(*args, **kwargs)
  File "c:\miniconda\envs\spikesort\Lib\site-packages\matplotlib\widgets.py", line 184, in wrapper
    if event.inaxes is not self.ax:
       ^^^^^^^^^^^^
AttributeError: 'ResizeEvent' object has no attribute 'inaxes'
Traceback (most recent call last):
  File "c:\miniconda\envs\spikesort\Lib\site-packages\matplotlib\cbook.py", line 390, in process
    func(*args, **kwargs)
  File "c:\miniconda\envs\spikesort\Lib\site-packages\matplotlib\widgets.py", line 184, in wrapper
    if event.inaxes is not self.ax:
       ^^^^^^^^^^^^
AttributeError: 'ResizeEvent' object has no attribute 'inaxes'
Traceback (most recent call last):
  File "c:\miniconda\envs\spikesort\Lib\site-packages\matplotlib\cbook.py", line 390, in process
    func(*args, **kwargs)
  File "c:\miniconda\envs\spikesort\Lib\site-packages\matplotlib\widgets.py", line 184, in